In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

def check_pin_visibility(image_path):
    # 1. Load Image
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian Blur (Common step)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    """ # Global Mean
    # "Find mean value"
    mean_val = np.mean(blurred)
    # "Reset to white if > mean" (Threshold)
    _, global_thresh = cv2.threshold(blurred, mean_val, 255, cv2.THRESH_BINARY)
    # "Binary Invert"
    user_result = cv2.bitwise_not(global_thresh) """

    # Adaptive Threshold
    # Block Size 25, Constant 10 (Fine tune these!)
    adaptive_result = cv2.adaptiveThreshold(blurred, 255, 
                                            cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                            cv2.THRESH_BINARY_INV, 45, 15)

    if os.path.exists("output"):
        """ cv2.imwrite(f"output/global_{image_path.split('/')[-1]}", user_result) """
        cv2.imwrite(f"output/adaptive_{image_path.split('/')[-1]}", adaptive_result)
    else:
        os.mkdir("output")
        """cv2.imwrite(f"output/global_{image_path.split('/')[-1]}", user_result) """
        cv2.imwrite(f"output/adaptive_{image_path.split('/')[-1]}", adaptive_result)

    # 3. Visualize
    """ plt.figure(figsize=(12, 6)) """
    
    """ plt.subplot(1, 2, 1)
    plt.imshow(user_result, cmap='gray')
    plt.title(f"User Method\n(Global Mean: {mean_val:.1f})")
    plt.axis('off') """

    """ plt.subplot(1, 2, 2) """
    """ plt.imshow(adaptive_result, cmap='gray')
    plt.title("Adaptive Threshold\n(Local Contrast)")
    plt.axis('off')

    plt.tight_layout()
    plt.show() """

for img in os.listdir("./data/socket/"):
    """ print(f"{img}:") """
    check_pin_visibility(f"./data/socket/{img}")

In [ ]:

def extract_grid_map(binary_img_path):
    # Load the binary image you just uploaded
    binary = cv2.imread(binary_img_path, cv2.IMREAD_GRAYSCALE)
    
    # 1. Morphological Close to fill "Donuts"
    # A 3x3 kernel is usually enough to bridge the gaps in the rings
    kernel = np.ones((3,3), np.uint8)
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    # 2. Find Contours
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 3. Filter and Extract Centroids
    valid_pins = []
    min_area = 300  # Filter out the tiny noise specks
    max_area = 700 # Filter out large artifacts if any
    
    vis_img = cv2.cvtColor(closed, cv2.COLOR_GRAY2BGR)
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if min_area < area < max_area:
            # Calculate Centroid (Center of Mass)
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
                valid_pins.append((cX, cY))
                
                # Visualize
                cv2.circle(vis_img, (cX, cY), 2, (0, 0, 255), -1)

    print(f"Mapped {len(valid_pins)} pins.")
    
    plt.figure(figsize=(10,10))
    plt.imshow(vis_img)
    plt.title(f"Grid Map: {len(valid_pins)} Pins Detected")
    plt.show()
    
    return valid_pins

In [ ]:
# Use the coordinates we found in the previous step
# coords = [(x1,y1), (x2,y2), ...] 

def create_anomalib_dataset(image_path, coords, output_dir="datasets/socket_pins"):
    img = cv2.imread(image_path)
    
    # Create folders
    train_dir = os.path.join(output_dir, "train/good")
    test_dir = os.path.join(output_dir, "test/good")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    # Crop settings
    crop_size = 32  # 32x32 is standard for small features like pins
    half = crop_size // 2
    
    for i, (cx, cy) in enumerate(coords):
        # Boundary check
        if cy-half < 0 or cx-half < 0: continue
        
        crop = img[cy-half:cy+half, cx-half:cx+half]
        
        # Save 80% to Train, 20% to Test
        if i % 5 == 0:
            cv2.imwrite(f"{test_dir}/pin_{i}.png", crop)
        else:
            cv2.imwrite(f"{train_dir}/pin_{i}.png", crop)
            
    print(f"Dataset generated at {output_dir}")

for img in os.listdir("./output/"):
    coords = extract_grid_map(f"./output/{img}")
    create_anomalib_dataset(f"./output/{img}", coords)